# PositionManager and PositionTracker Testing

This notebook provides a comprehensive test suite for both `PositionManager` and `PositionTracker` modules within the `PositionManager` directory. It uses mocks to simulate MetaTrader 5 (MT5) interactions, allowing the tests to run without a live MT5 connection.

In [1]:
import sys
import os
import time
import json
import logging
from datetime import datetime, timezone
from unittest.mock import MagicMock, patch

# Add current directory to path so we can import the modules
sys.path.append(os.getcwd())

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(name)s: %(message)s')
logger = logging.getLogger("TestNotebook")

## 1. Mocking MetaTrader 5

Since MT5 is a Windows-only library and requires a live terminal, we mock it entirely for these tests.

In [2]:
# Create a mock for the MetaTrader5 module
mt5 = MagicMock()

# Define necessary constants
mt5.ORDER_FILLING_FOK = 0
mt5.ORDER_TYPE_BUY = 0
mt5.ORDER_TYPE_SELL = 1
mt5.TRADE_ACTION_DEAL = 1
mt5.TRADE_ACTION_SLTP = 6
mt5.ORDER_TIME_GTC = 0
mt5.TRADE_RETCODE_DONE = 10009
mt5.TRADE_RETCODE_REQUOTE = 10004

# Inject the mock into sys.modules
sys.modules["MetaTrader5"] = mt5

print("MetaTrader5 mocked successfully.")

MetaTrader5 mocked successfully.


## 2. Testing PositionManager

The `PositionManager` is a pure executor. We'll test its `open_position`, `close_position`, and `modify_position` methods.

In [3]:
from position_manager import PositionManager

def test_position_manager_open():
    print("--- Testing PositionManager.open_position ---")
    pm = PositionManager(magic_unity=100001, magic_mm=100002)
    
    # Mock symbol info tick
    mt5.symbol_info_tick.return_value = MagicMock(ask=1.1000, bid=1.0990)
    
    # Mock successful order send
    mock_result = MagicMock()
    mock_result.retcode = mt5.TRADE_RETCODE_DONE
    mock_result.order = 123456
    mock_result.price = 1.1000
    mock_result.comment = "Request executed"
    mt5.order_send.return_value = mock_result
    mt5.last_error.return_value = (1, "Success")
    
    result = pm.open_position(
        symbol="GBPUSD_i",
        direction=1,
        lot_size=0.1,
        sl_price=1.0900,
        tp_price=1.1100,
        strategy="unity"
    )
    
    print(f"Open Result: {json.dumps(result, indent=2, default=str)}")
    assert result["success"] is True
    assert result["ticket"] == 123456
    assert result["strategy"] == "unity"
    assert result["lot_size"] == 0.1
    assert result["error_code"] == 0
    print("Open position test passed!\n")

def test_position_manager_close():
    print("--- Testing PositionManager.close_position ---")
    pm = PositionManager(magic_unity=100001, magic_mm=100002)
    
    # Mock position to close
    mock_pos = MagicMock()
    mock_pos.symbol = "GBPUSD_i"
    mock_pos.volume = 0.1
    mock_pos.type = mt5.ORDER_TYPE_BUY
    mock_pos.magic = 100001
    mock_pos.sl = 1.0900
    mock_pos.tp = 1.1100
    mt5.positions_get.return_value = [mock_pos]
    
    # Mock successful close
    mock_result = MagicMock()
    mock_result.retcode = mt5.TRADE_RETCODE_DONE
    mock_result.price = 1.1005
    mock_result.comment = "Closed"
    mt5.order_send.return_value = mock_result
    mt5.symbol_info_tick.return_value = MagicMock(bid=1.1005, ask=1.1006)
    
    result = pm.close_position(ticket=123456)
    
    print(f"Close Result: {json.dumps(result, indent=2, default=str)}")
    assert result["success"] is True
    assert result["lot_size"] == 0.1
    print("Close position test passed!\n")

test_position_manager_open()
test_position_manager_close()

2026-06-18 13:29:02,874 [INFO] PositionManager: PositionManager initialized (magic_unity=100001, magic_mm=100002)
2026-06-18 13:29:02,875 [INFO] PositionManager: OPEN SUCCESS: GBPUSD_i unity 1 0.1 SL:1.09 TP:1.11 Ticket:123456
2026-06-18 13:29:02,875 [INFO] PositionManager: PositionManager initialized (magic_unity=100001, magic_mm=100002)
2026-06-18 13:29:02,878 [INFO] PositionManager: CLOSE SUCCESS: Ticket 123456 Symbol GBPUSD_i Vol 0.1


--- Testing PositionManager.open_position ---
Open Result: {
  "success": true,
  "ticket": 123456,
  "symbol": "GBPUSD_i",
  "direction": 1,
  "lot_size": 0.1,
  "entry_price": 1.1,
  "sl_price": 1.09,
  "tp_price": 1.11,
  "strategy": "unity",
  "error_code": 0,
  "retcode": 10009,
  "comment": ""
}
Open position test passed!

--- Testing PositionManager.close_position ---
Close Result: {
  "success": true,
  "ticket": 123456,
  "symbol": "GBPUSD_i",
  "direction": 1,
  "lot_size": 0.1,
  "entry_price": 1.1005,
  "sl_price": 1.09,
  "tp_price": 1.11,
  "strategy": "unity",
  "error_code": 0,
  "retcode": 10009,
  "comment": "Success"
}
Close position test passed!



## 3. Testing PositionTracker

The `PositionTracker` is an observer that polls MT5 in a background thread. We'll test its ability to track positions, calculate risk/reward, and persist state.

In [4]:
from position_tracker import PositionTracker
from drawdown import DrawdownManager

def test_position_tracker():
    print("--- Testing PositionTracker ---")
    state_file = "test_position_state_notebook.json"
    if os.path.exists(state_file): os.remove(state_file)
    
    tracker = PositionTracker(
        magic_numbers=[100001, 100002],
        poll_interval_seconds=1,
        state_file=state_file
    )
    
    # Mock positions for tracker
    mock_pos1 = MagicMock()
    mock_pos1.ticket = 111
    mock_pos1.symbol = "GBPUSD_i"
    mock_pos1.magic = 100001
    mock_pos1.type = mt5.ORDER_TYPE_BUY
    mock_pos1.volume = 0.1
    mock_pos1.price_open = 1.1000
    mock_pos1.sl = 1.0950 # 50 pips risk
    mock_pos1.tp = 1.1100 # 100 pips reward
    mock_pos1.price_current = 1.1010
    mock_pos1.profit = 10.0
    mock_pos1.time = time.time()
    
    mt5.positions_get.return_value = [mock_pos1]
    
    # Mock symbol info for contract size
    mock_info = MagicMock()
    mock_info.trade_contract_size = 100000
    mt5.symbol_info.return_value = mock_info
    
    tracker.start()
    time.sleep(1.5) # Allow one poll cycle
    
    positions = tracker.get_open_positions()
    risk = tracker.get_open_risk()
    reward = tracker.get_open_reward()
    
    print(f"Tracked Positions: {len(positions)}")
    print(f"Total Risk: ${risk:.2f}")
    print(f"Total Reward: ${reward:.2f}")
    
    # Expected Risk: 0.0050 * 0.1 * 100000 = 50.0
    # Expected Reward: 0.0100 * 0.1 * 100000 = 100.0
    assert len(positions) == 1
    assert abs(risk - 50.0) < 0.01
    assert abs(reward - 100.0) < 0.01
    assert isinstance(positions[0]["open_time"], str)
    
    tracker.stop()
    
    # Verify persistence
    assert os.path.exists(state_file)
    with open(state_file, "r") as f:
        saved_data = json.load(f)
        assert len(saved_data["positions"]) == 1
        print("State file persisted correctly.")
    
    if os.path.exists(state_file): os.remove(state_file)
    print("PositionTracker test passed!\n")

test_position_tracker()

2026-06-18 13:29:03,529 [INFO] PositionTracker: New tracked position detected: ticket 111
2026-06-18 13:29:03,530 [INFO] PositionTracker: PositionTracker started.
2026-06-18 13:29:03,533 [INFO] PositionTracker: Poll cycle summary: 1 positions tracked, total open risk $50.00


--- Testing PositionTracker ---


2026-06-18 13:29:04,547 [INFO] PositionTracker: Poll cycle summary: 1 positions tracked, total open risk $50.00
2026-06-18 13:29:05,033 [INFO] PositionTracker: PositionTracker stopped.


Tracked Positions: 1
Total Risk: $50.00
Total Reward: $100.00
State file persisted correctly.
PositionTracker test passed!



## 5. Testing DrawdownManager

The `DrawdownManager` enforces daily and total risk limits based on account balance and `PositionTracker` risk.

In [ ]:
def test_drawdown_manager():
    print("--- Testing DrawdownManager ---")
    state_file = "test_drawdown_state_notebook.json"
    if os.path.exists(state_file): os.remove(state_file)
    
    # Mock PositionTracker
    mock_tracker = MagicMock()
    mock_tracker.get_open_positions.return_value = []
    mock_tracker.get_open_risk.return_value = 0.0
    
    # Mock MT5 Account and Time
    mt5.account_info.return_value = MagicMock(balance=10000.0)
    mt5.symbol_info_tick.return_value = MagicMock(time=time.mktime(time.strptime("2024-05-20", "%Y-%m-%d")))
    mt5.symbol_select.return_value = True
    
    dm = DrawdownManager(
        initial_balance=10000.0,
        position_tracker=mock_tracker,
        daily_limit_pct=0.03,
        total_limit_pct=0.10,
        state_file=state_file
    )
    
    print(f"Initial: Allowed={dm.trading_allowed()}, MaxRisk={dm.max_risk_pct():.4%}")
    assert dm.trading_allowed() == True
    
    # Scenario: $200 risk
    mock_tracker.get_open_risk.return_value = 200.0
    dm.check()
    print(f"Post-Risk: Allowed={dm.trading_allowed()}, MaxRisk={dm.max_risk_pct():.4%}")
    assert abs(dm.max_risk_pct() - 0.01) < 1e-6
    
    # Scenario: $200 risk + $150 loss
    mt5.account_info.return_value = MagicMock(balance=9850.0)
    dm.check()
    print(f"Post-Loss: Allowed={dm.trading_allowed()}, MaxRisk={dm.max_risk_pct():.4%}")
    assert dm.trading_allowed() == False
    
    if os.path.exists(state_file): os.remove(state_file)
    print("DrawdownManager test passed!\n")

test_drawdown_manager()

## 4. Error Handling and Edge Cases

Testing how modules handle invalid inputs and MT5 errors.

In [5]:
def test_edge_cases():
    print("--- Testing Edge Cases ---")
    pm = PositionManager(magic_unity=100001, magic_mm=100002)
    
    # 1. Invalid Direction
    res1 = pm.open_position("GBPUSD_i", 5, 0.1, 1.1, 1.2, "unity")
    assert res1["success"] is False
    assert "Invalid direction" in res1["comment"]
    
    # 2. MT5 Requote
    mt5.symbol_info_tick.return_value = MagicMock(ask=1.1000, bid=1.0990)
    mock_fail = MagicMock()
    mock_fail.retcode = mt5.TRADE_RETCODE_REQUOTE
    mock_fail.comment = "Requote"
    mt5.order_send.return_value = mock_fail
    
    res2 = pm.open_position("GBPUSD_i", 1, 0.1, 1.09, 1.11, "unity")
    assert res2["success"] is False
    assert res2["retcode"] == mt5.TRADE_RETCODE_REQUOTE
    
    print("Edge cases test passed!\n")

test_edge_cases()

2026-06-18 13:29:05,052 [INFO] PositionManager: PositionManager initialized (magic_unity=100001, magic_mm=100002)
2026-06-18 13:29:05,052 [ERROR] PositionManager: OPEN FAILED: GBPUSD_i unity 1 Lot:0.1 Retcode:10004 Comment:Requote


--- Testing Edge Cases ---
Edge cases test passed!



In [6]:
def test_position_manager_modify():
    print("--- Testing PositionManager.modify_position ---")
    pm = PositionManager(magic_unity=100001, magic_mm=100002)

    mock_pos = MagicMock()
    mock_pos.symbol = "GBPUSD_i"
    mock_pos.magic = 100001
    mock_pos.type = mt5.ORDER_TYPE_BUY
    mock_pos.volume = 0.1
    mock_pos.price_open = 1.1000
    mock_pos.sl = 1.0900
    mock_pos.tp = 1.1100
    mt5.positions_get.return_value = [mock_pos]

    mock_result = MagicMock()
    mock_result.retcode = mt5.TRADE_RETCODE_DONE
    mock_result.comment = "Modified"
    mt5.order_send.return_value = mock_result

    # Test 1: modify SL only — TP should be preserved from MT5
    result = pm.modify_position(ticket=123456, sl_price=1.0920)
    assert result["success"] is True
    assert result["sl_price"] == 1.0920
    assert result["tp_price"] == 1.1100  # preserved from pos.tp
    
    # Verify the actual request sent to MT5 used the preserved TP
    call_args = mt5.order_send.call_args[0][0]
    assert call_args["sl"] == 1.0920
    assert call_args["tp"] == 1.1100

    # Test 2: modify TP only — SL preserved
    result = pm.modify_position(ticket=123456, tp_price=1.1200)
    call_args = mt5.order_send.call_args[0][0]
    assert call_args["sl"] == 1.0900  # preserved
    assert call_args["tp"] == 1.1200

    print("Modify position test passed!\n")

In [7]:
test_position_manager_modify()

2026-06-18 13:31:43,630 [INFO] PositionManager: PositionManager initialized (magic_unity=100001, magic_mm=100002)
2026-06-18 13:31:43,631 [INFO] PositionManager: MODIFY SUCCESS: Ticket 123456 SL:1.092 TP:1.11
2026-06-18 13:31:43,632 [INFO] PositionManager: MODIFY SUCCESS: Ticket 123456 SL:1.09 TP:1.12


--- Testing PositionManager.modify_position ---
Modify position test passed!

